In [ ]:
"""
Be-10 Burial Dating and Paleo-Erosion Rate Calculator
======================================================

This script calculates paleo-erosion rates from Be-10 measurements in buried 
fluvial sediments, accounting for radioactive decay and paleomagnetic field 
variations.

Key References:
---------------
- Granger (2006): Burial dating methods, GSA Special Paper 415
- Chmeleff et al. (2010) & Korschinek et al. (2010): Be-10 half-life = 1.387 Ma
  (Nuclear Instruments and Methods in Physics Research B, 268)
- Lifton et al. (2014): Time-dependent production rate scaling
  (Quaternary Geochronology, 26, 56-69)
- Stone (2000): Atmospheric pressure and cosmogenic isotope production
  (JGR, 105, 23753-23759)
- Balco et al. (2008): CRONUS-Earth calculator methods
  (Quaternary Geochronology, 3, 174-195)
"""

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# INPUT DATA - MODIFY THIS SECTION WITH YOUR DATA
# =============================================================================

# Buried fluvial samples (paleo-erosion rates)
# These samples were exposed at the surface (accumulating Be-10), then buried
# (stopping production and allowing decay)
paleo_samples = {
    'CT-Ter-2': {
        'lat': -35.057,          # Latitude (degrees, negative = south)
        'lon': -72.164,          # Longitude (degrees, negative = west)
        'elev': 23,              # Elevation (meters above sea level)
        'Be10': 5.10e4,          # Measured Be-10 concentration (atoms/g)
        'Be10_err': 9.67e2,      # Be-10 measurement error (atoms/g, 1-sigma)
        'age': 1.0,              # Burial age (Ma) from independent dating
        'age_err': 0.17          # Burial age uncertainty (Ma, 1-sigma)
    },
    'CT-Ter-10': {
        'lat': -35.057,
        'lon': -72.164,
        'elev': 53,
        'Be10': 1.20e5,
        'Be10_err': 2.25e3,
        'age': 0.5,
        'age_err': 0.09
    }
}

# Modern erosion rate samples (for comparison)
# These provide present-day erosion rates to compare with paleo rates
modern_samples = {
    'CT-9': {
        'lat': -35.046389,       # Basin outlet location
        'lon': -72.101111,
        'eros': 105,             # Modern erosion rate (m/Myr)
        'err': 10,               # Uncertainty (m/Myr, 1-sigma)
        'type': 'basin'          # Basin-integrated rate
    },
    'CT-3': {
        'lat': -35.10038,        # Coastal cordillera location
        'lon': -71.94672,
        'eros': 29.5,            # Modern erosion rate (m/Myr)
        'err': 5,                # Uncertainty (m/Myr, 1-sigma)
        'type': 'hillslope'      # Local hillslope rate
    }
}

# Physical constants and parameters
HALFLIFE = 1.387          # Be-10 half-life in Ma (Chmeleff et al. 2010; Korschinek et al. 2010)
DENSITY = 2.6             # Rock density (g/cm³) for quartz-bearing rock
ATTEN_LENGTH = 160        # Cosmic ray attenuation length (g/cm²) from Gosse & Phillips (2001)
P0_SLHL = 4.0            # Sea-level high-latitude production rate (atoms/g/yr) from Stone (2000)

# =============================================================================
# CORE CALCULATION FUNCTIONS
# =============================================================================

def decay_constant(halflife):
    """
    Calculate radioactive decay constant from half-life.
    
    λ = ln(2) / t_1/2
    
    Parameters:
    -----------
    halflife : float
        Half-life in millions of years
    
    Returns:
    --------
    lambda : float
        Decay constant in Ma^-1
    
    Reference: Basic radioactive decay physics
    """
    return np.log(2) / halflife


def decay_correction(N_measured, burial_age, lambda_decay):
    """
    Correct measured Be-10 for radioactive decay during burial.
    
    The fundamental equation for radioactive decay is:
    N(t) = N₀ × e^(-λt)
    
    Rearranging to solve for original concentration:
    N₀ = N(t) × e^(λt)
    
    Parameters:
    -----------
    N_measured : float
        Measured Be-10 concentration today (atoms/g)
    burial_age : float
        Time since burial (Ma)
    lambda_decay : float
        Decay constant (Ma^-1)
    
    Returns:
    --------
    N_burial : float
        Be-10 concentration at time of burial (atoms/g)
    
    Reference: Granger (2006) equation 1
    """
    return N_measured * np.exp(lambda_decay * burial_age)


def paleomag_correction(burial_age):
    """
    Estimate production rate correction due to paleomagnetic field variations.
    
    Earth's magnetic field has varied significantly over the Quaternary. Weaker
    magnetic field → less cosmic ray shielding → higher production rates.
    
    These corrections are based on paleomagnetic databases (PADM2M, GGF100k)
    showing field intensity changes through time.
    
    Parameters:
    -----------
    burial_age : float
        Time since burial (Ma)
    
    Returns:
    --------
    correction_factor : float
        Multiplicative correction for production rate
        (1.0 = same as modern, >1.0 = higher than modern)
    
    Reference: Lifton et al. (2014) for time-dependent scaling
               Paleomag databases: Ziegler et al. (2011), Panovska et al. (2018)
    
    Notes:
    ------
    - 0-0.3 Ma: Field similar to modern (correction = 1.0)
    - 0.3-0.6 Ma: Mid-Brunhes, slightly weaker field (correction = 1.07)
    - 0.6-0.78 Ma: Approaching Brunhes-Matuyama reversal, weaker (correction = 1.10)
    - 0.78-1.2 Ma: Early Brunhes, weaker field (correction = 1.12)
    - >1.2 Ma: Matuyama, variable field (correction = 1.08)
    """
    if burial_age <= 0.3:
        return 1.00  # Modern field strength
    elif burial_age <= 0.6:
        return 1.07  # Mid-Brunhes weak field
    elif burial_age <= 0.78:
        return 1.10  # Approaching B-M reversal
    elif burial_age <= 1.2:
        return 1.12  # Early Brunhes
    else:
        return 1.08  # Matuyama chron


def stone_scaling(latitude, elevation, paleo_correction=1.0):
    """
    Calculate production rate scaling factor using Stone (2000) formulation.
    
    The production rate of cosmogenic nuclides varies with:
    1. Elevation (atmospheric pressure) - exponential relationship
    2. Latitude (geomagnetic field strength) - polynomial fit
    
    This function implements the Stone (2000) time-independent scaling scheme.
    
    Parameters:
    -----------
    latitude : float
        Geographic latitude in degrees (negative for southern hemisphere)
    elevation : float
        Elevation in meters above sea level
    paleo_correction : float
        Paleomagnetic correction factor (default 1.0 = modern field)
    
    Returns:
    --------
    scaling_factor : float
        Multiplicative factor for SLHL production rate
    
    Reference: Stone (2000) JGR 105, equations 1-3 and Table 1
    """
    # Step 1: Calculate pressure scaling
    # Atmospheric pressure decreases exponentially with altitude
    # P(z) = P₀ × exp(-z/H) where H ≈ 8400 m is the scale height
    P_sea = 1013.25  # Sea level pressure (hPa)
    H = 8400         # Atmospheric scale height (m)
    P_elev = P_sea * np.exp(-elevation / H)
    
    # Production rate scales exponentially with pressure difference
    # This accounts for the fact that higher elevations have less atmospheric
    # shielding and therefore higher cosmic ray flux
    pressure_factor = np.exp((P_sea - P_elev) / 150.0)
    
    # Step 2: Calculate latitude scaling
    # Geomagnetic field is stronger at poles, weaker at equator
    # Stone (2000) uses a polynomial fit to capture this variation
    lat_abs = np.abs(latitude)
    
    # Polynomial coefficients from Stone (2000) Table 1
    a = 31.8518
    b = 250.3193
    c = -0.083393
    d = 7.4260e-5
    e = -2.2397e-8
    
    # Calculate latitude-dependent scaling
    lat_scaling = a + b * np.exp(c * lat_abs + d * lat_abs**2 + e * lat_abs**3)
    lat_factor = lat_scaling / a  # Normalize to equator
    
    # Step 3: Combine all factors
    return lat_factor * pressure_factor * paleo_correction


def production_rate(latitude, elevation, burial_age):
    """
    Calculate paleo-production rate at sample location and time.
    
    This combines the modern SLHL production rate with:
    1. Latitude/elevation scaling (Stone 2000)
    2. Paleomagnetic field corrections (Lifton et al. 2014)
    
    Parameters:
    -----------
    latitude : float
        Geographic latitude (degrees)
    elevation : float
        Elevation (meters asl)
    burial_age : float
        Time since burial (Ma) - used for paleomagnetic correction
    
    Returns:
    --------
    P_paleo : float
        Paleo-production rate at sample location (atoms/g/yr)
    
    Reference: Balco et al. (2008) - CRONUS calculator methodology
    """
    # Get paleomagnetic correction for the burial age
    paleo_corr = paleomag_correction(burial_age)
    
    # Calculate geographic scaling factor
    scaling = stone_scaling(latitude, elevation, paleo_corr)
    
    # Apply to SLHL production rate
    return P0_SLHL * scaling


def erosion_rate(N_burial, P_paleo, lambda_decay, atten_length=160, density=2.7):
    """
    Calculate steady-state erosion rate from Be-10 concentration.
    
    At steady-state with erosion, Be-10 production is balanced by decay and
    removal through erosion:
    
    Production = Decay + Erosion loss
    P = λN + (ε/Λ)N
    
    Rearranging:
    N = (P × Λ) / (λ + ε/Λ)
    
    Solving for erosion rate ε:
    ε = Λ × [(P/N) - λ]
    
    Parameters:
    -----------
    N_burial : float
        Be-10 concentration at burial (atoms/g)
    P_paleo : float
        Paleo-production rate (atoms/g/yr)
    lambda_decay : float
        Decay constant (Ma^-1)
    atten_length : float
        Cosmic ray attenuation length (g/cm²), default 160
    density : float
        Rock density (g/cm³), default 2.7
    
    Returns:
    --------
    erosion_m_Myr : float
        Erosion rate in meters per million years
    
    Reference: Lal (1991), Brown et al. (1995), Granger et al. (1996)
    
    Notes:
    ------
    The attenuation length Λ describes how quickly cosmic rays are absorbed
    by rock. A typical value for rock is ~160 g/cm².
    """
    # Convert decay constant from Ma^-1 to yr^-1
    lambda_yr = lambda_decay * 1e-6
    
    # Calculate erosion rate in g/cm²/yr
    erosion_gcm2_yr = atten_length * ((P_paleo / N_burial) - lambda_yr)
    
    # Convert to physical erosion rate (m/Myr)
    # Step 1: Convert from g/cm²/yr to cm/yr by dividing by density
    erosion_cm_yr = erosion_gcm2_yr / density
    
    # Step 2: Convert cm/yr to m/Myr
    # cm/yr × 10 → mm/yr
    # mm/yr × 1000 → m/Myr
    erosion_m_Myr = erosion_cm_yr * 10 * 1000
    
    return erosion_m_Myr


def propagate_error(sample_data, lambda_decay):
    """
    Propagate analytical uncertainties through all calculations.
    
    This function uses simple error propagation to estimate how uncertainties
    in Be-10 concentration and burial age affect the calculated erosion rate.
    
    Method:
    -------
    1. Calculate erosion rate with central values
    2. Calculate erosion rate with Be-10 ± 1σ
    3. Calculate erosion rate with age ± 1σ
    4. Combine uncertainties in quadrature
    
    Parameters:
    -----------
    sample_data : dict
        Dictionary containing sample measurements and uncertainties
    lambda_decay : float
        Decay constant (Ma^-1)
    
    Returns:
    --------
    total_error : float
        Total 1-sigma uncertainty in erosion rate (m/Myr)
    
    Reference: Balco et al. (2008) - error propagation methods
    """
    # Extract data
    lat = sample_data['lat']
    elev = sample_data['elev']
    Be10 = sample_data['Be10']
    Be10_err = sample_data['Be10_err']
    age = sample_data['age']
    age_err = sample_data['age_err']
    
    # Central value calculation
    N_burial_central = decay_correction(Be10, age, lambda_decay)
    P_central = production_rate(lat, elev, age)
    E_central = erosion_rate(N_burial_central, P_central, lambda_decay, 
                            ATTEN_LENGTH, DENSITY)
    
    # Vary Be-10 concentration by ±1σ
    # Note: Higher Be-10 → Lower erosion rate (inverse relationship)
    N_burial_high = decay_correction(Be10 + Be10_err, age, lambda_decay)
    N_burial_low = decay_correction(Be10 - Be10_err, age, lambda_decay)
    
    E_high_Be10 = erosion_rate(N_burial_low, P_central, lambda_decay, 
                               ATTEN_LENGTH, DENSITY)
    E_low_Be10 = erosion_rate(N_burial_high, P_central, lambda_decay, 
                              ATTEN_LENGTH, DENSITY)
    
    # Half-width of uncertainty from Be-10
    error_from_Be10 = (E_high_Be10 - E_low_Be10) / 2
    
    # Vary burial age by ±1σ
    # Older age → more decay correction → higher burial concentration → lower erosion
    N_burial_age_high = decay_correction(Be10, age + age_err, lambda_decay)
    P_age_high = production_rate(lat, elev, age + age_err)
    E_age_high = erosion_rate(N_burial_age_high, P_age_high, lambda_decay,
                             ATTEN_LENGTH, DENSITY)
    
    N_burial_age_low = decay_correction(Be10, age - age_err, lambda_decay)
    P_age_low = production_rate(lat, elev, age - age_err)
    E_age_low = erosion_rate(N_burial_age_low, P_age_low, lambda_decay,
                            ATTEN_LENGTH, DENSITY)
    
    # Half-width of uncertainty from age
    error_from_age = (E_age_high - E_age_low) / 2
    
    # Combine uncertainties in quadrature (assumes independent errors)
    total_error = np.sqrt(error_from_Be10**2 + error_from_age**2)
    
    return total_error


# =============================================================================
# MAIN CALCULATION WORKFLOW
# =============================================================================

# Calculate decay constant from half-life
lambda_decay = decay_constant(HALFLIFE)

# Initialize results storage
results = []

# Print header
print("\n" + "="*70)
print("BE-10 PALEO-EROSION RATE ANALYSIS")
print("="*70)
print(f"Using Be-10 half-life: {HALFLIFE} Ma")
print(f"Rock density: {DENSITY} g/cm³")
print(f"Attenuation length: {ATTEN_LENGTH} g/cm²")
print(f"SLHL production rate: {P0_SLHL} atoms/g/yr")

# Process each sample
for sample_name, data in paleo_samples.items():
    print(f"\n{sample_name}:")
    
    # Step 1: Apply decay correction
    # This accounts for Be-10 lost to radioactive decay during burial
    N_burial = decay_correction(data['Be10'], data['age'], lambda_decay)
    decay_factor = N_burial / data['Be10']
    percent_lost = (1 - 1/decay_factor) * 100
    
    # Step 2: Calculate paleo-production rate
    # This accounts for paleomagnetic field variations and geographic location
    P_paleo = production_rate(data['lat'], data['elev'], data['age'])
    paleo_corr = paleomag_correction(data['age'])
    
    # Step 3: Calculate erosion rate
    # This uses the steady-state erosion equation
    E = erosion_rate(N_burial, P_paleo, lambda_decay, ATTEN_LENGTH, DENSITY)
    
    # Step 4: Propagate uncertainties
    E_err = propagate_error(data, lambda_decay)
    
    # Print results for this sample
    print(f"  Measured [Be-10]: {data['Be10']:.2e} atoms/g")
    print(f"  Burial [Be-10]: {N_burial:.2e} atoms/g")
    print(f"  Decay correction: ×{decay_factor:.3f} ({percent_lost:.1f}% lost)")
    print(f"  Paleomag correction: ×{paleo_corr:.3f}")
    print(f"  Paleo production rate: {P_paleo:.2f} atoms/g/yr")
    print(f"  Erosion rate: {E:.1f} ± {E_err:.1f} m/Myr")
    
    # Store results
    results.append({
        'Sample': sample_name,
        'Age_Ma': data['age'],
        'Be10_burial': N_burial,
        'Prod_Rate': P_paleo,
        'Erosion_mMyr': E,
        'Error_mMyr': E_err,
        'Lat': data['lat'],
        'Lon': data['lon']
    })

# Convert results to DataFrame and save
df = pd.DataFrame(results)
df.to_csv('be10_results.csv', index=False, float_format='%.4f')

print(f"\n{'='*70}")
print("Results saved to: be10_results.csv")


BE-10 PALEO-EROSION RATE ANALYSIS
Using Be-10 half-life: 1.387 Ma
Rock density: 2.6 g/cm³
Attenuation length: 160 g/cm²
SLHL production rate: 4.0 atoms/g/yr

CT-Ter-2:
  Measured [Be-10]: 5.10e+04 atoms/g
  Burial [Be-10]: 8.41e+04 atoms/g
  Decay correction: ×1.648 (39.3% lost)
  Paleomag correction: ×1.120
  Paleo production rate: 6.67 atoms/g/yr
  Erosion rate: 48.5 ± 4.3 m/Myr

CT-Ter-10:
  Measured [Be-10]: 1.20e+05 atoms/g
  Burial [Be-10]: 1.54e+05 atoms/g
  Decay correction: ×1.284 (22.1% lost)
  Paleomag correction: ×1.070
  Paleo production rate: 6.53 atoms/g/yr
  Erosion rate: 25.8 ± 1.3 m/Myr

Results saved to: be10_results.csv
